In [1]:
import pandas as pd
pred = pd.read_csv("../data_processed/predictions_hgb_hierarchical_2022.csv")
pred["invoice_date"] = pd.to_datetime(pred["invoice_date"])
pred.head()

,store_id,invoice_date,invoice_count,oc_count,fleet_oc_count,non_fleet_oc,severity,pred_invoice,pred_non_fleet_oc,pred_fleet_oc,pred_oc_total,abs_err_invoice,abs_err_oc,oc_lower_95,oc_upper_95
0,79609,2022-01-02,37,31,3,28,sev_freezing,34.904848,26.133422,0.832786,26.966207,2.095152,4.033793,11.326312,42.529751
1,79609,2022-01-03,60,55,7,48,sev_freezing,58.718083,43.449059,5.084164,48.533223,1.281917,6.466777,32.893328,64.096767
2,79609,2022-01-04,62,50,5,45,sev_freezing,52.945849,40.869569,5.170848,46.040417,9.054151,3.959583,30.400522,61.603961
3,79609,2022-01-05,52,44,4,40,sev_freezing,53.302560,37.263945,4.685428,41.949373,1.302560,2.050627,26.309478,57.512918
4,79609,2022-01-06,16,13,1,12,sev_freezing,40.850080,29.404225,4.278599,33.682824,24.850080,20.682824,18.042930,49.246369


In [2]:
pred["abs_error"] = (pred["oc_count"] - pred["pred_oc_total"]).abs()

pred["dow"] = pred["invoice_date"].dt.dayofweek
pred["month"] = pred["invoice_date"].dt.month

In [3]:
worst = pred.sort_values("abs_error", ascending=False).head(50)

worst[[
    "store_id",
    "invoice_date",
    "oc_count",
    "pred_oc_total",
    "abs_error",
    "severity"
]]

,store_id,invoice_date,oc_count,pred_oc_total,abs_error,severity
34495,102098,2022-05-20,220,44.479734,175.520266,sev_normal
59053,232116,2022-05-10,166,39.394574,126.605426,sev_normal
35110,102302,2022-03-18,200,85.559977,114.440023,sev_normal
6878,86960,2022-08-01,170,57.554761,112.445239,sev_rain_heavy
4396,84970,2022-07-29,175,66.057859,108.942141,sev_normal
59063,232116,2022-05-20,174,70.902068,103.097932,sev_normal
11349,88666,2022-02-12,160,58.323915,101.676085,sev_freezing
102702,611637,2022-04-20,136,34.659177,101.340823,sev_normal
54181,230659,2022-07-26,132,31.410030,100.589970,sev_normal
52334,230652,2022-02-14,153,57.743714,95.256286,sev_snow_heavy


In [4]:
dow_error = pred.groupby("dow")["abs_error"].mean().sort_values(ascending=False)
dow_error

dow
4    6.077800
5    5.916734
3    5.726752
1    5.621217
0    5.544217
2    5.498515
6    5.228090
Name: abs_error, dtype: float64

In [5]:
month_error = pred.groupby("month")["abs_error"].mean().sort_values(ascending=False)
month_error

month
2     6.527188
12    6.331334
1     6.093505
7     5.619614
11    5.584466
9     5.584115
5     5.547973
3     5.514950
6     5.444365
4     5.357708
8     5.301694
10    5.179590
Name: abs_error, dtype: float64

In [6]:
store_error = pred.groupby("store_id")["abs_error"].mean().sort_values(ascending=False).head(10)
store_error

store_id
99215     10.313214
231320    10.076084
255712     9.049198
230586     8.957205
84321      8.806127
100510     8.798450
100502     8.511222
86765      8.313630
232128     8.143036
594271     8.115688
Name: abs_error, dtype: float64

In [7]:
pred["volume_bucket"] = pd.cut(
    pred["oc_count"],
    bins=[0, 10, 30, 60, 100, 200],
    labels=["0-10", "10-30", "30-60", "60-100", "100+"]
)
volume_error = pred.groupby("volume_bucket")["abs_error"].mean()
volume_error

/var/folders/tg/304lj0vs4yn7768vztr026j80000gn/T/ipykernel_31799/811071242.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  volume_error = pred.groupby("volume_bucket")["abs_error"].mean()


volume_bucket
0-10      16.287371
10-30      5.725355
30-60      5.053495
60-100     6.996979
100+      22.814854
Name: abs_error, dtype: float64

In [8]:
severity_error = pred.groupby("severity")["abs_error"].mean().sort_values(ascending=False)
severity_error

severity
sev_cold_extreme    6.929449
sev_snow_heavy      6.478821
sev_rain_heavy      6.044632
sev_freezing        5.991864
sev_normal          5.437208
sev_heat_extreme    5.299430
Name: abs_error, dtype: float64

# Outlier Report — Worst Predictions (2022)

## Summary

We analyzed the top 50 largest prediction errors and performed breakdowns across time, store, volume, and weather severity to understand failure patterns.

---

## Key Findings

### 1. Errors are highest at extreme demand levels
- Very low volume (0–10 OC) → unstable predictions
- Very high volume (100+ OC) → large underprediction
- Indicates model struggles with both tails of distribution

---

### 2. High-traffic days are harder to predict
- Fridays show highest average error
- Suggests peak demand variability is not fully captured

---

### 3. Winter months have higher errors
- February, December, and January show highest error
- Likely due to weather volatility and seasonal effects

---

### 4. Certain stores consistently underperform
- Small subset of stores contributes disproportionately to error
- Indicates missing store-specific signals

---

### 5. Extreme weather increases prediction error
- Highest errors during:
  - Extreme cold
  - Heavy snow
  - Heavy rain
- Model struggles with nonlinear behavior under severe conditions

---

## Root Causes

- Lack of effectiveness in the holiday/closure features
- Insufficient modeling of extreme demand spikes
- Missing store-specific calibration
- Limited handling of low-volume instability

---

## Recommended Fixes

1. For holiday and special-event features  
   → capture abnormal spikes (e.g., holidays, promotions)

2. Introduce lag-based and nonlinear weather features  
   → improve modeling of extreme weather effects

3. Improve handling of demand extremes  
   → apply smoothing or separate modeling for low/high volume stores

---

## Conclusion

Model errors are not random; they are concentrated in extreme demand scenarios, winter months, specific stores, and severe weather conditions. Addressing these areas will significantly improve predictive accuracy.